In [ ]:
import csv
import numpy as np
from pathlib import Path
from skimage import io

# --- 1. Configuration & Paths ---
IMAGE_PATH = "C1-Lysosome_masked_crops_1024.tif"
WAVELET_DIR = Path("saved_wavelet_images")
CLUSTER_CSV = "Lysosome_annotations_combined.csv"

# --- 2. Load and Downsample Original Images ---
# Load multi-stack TIFF and downsample by factor of 2 in X and Y
images = io.imread(IMAGE_PATH)
n_images = images.shape[0]

# --- 3. Load Wavelet Filtered Images ---
# Loading multi-channel wavelet data
wavelet_images = [
    io.imread(WAVELET_DIR / f"wavelet_multi_channel_id_{i}.tif")
    for i in range(n_images)
]

# --- 4. Numeric and Descriptive Labels & Filtering ---
labels = []
valid_indices = []

with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    # Use DictReader for easier access to named columns
    reader = csv.DictReader(csvfile)
    
    for i, row in enumerate(reader):
        # 1. Filter out "NoClass" (annotation -1.0)
        if row['annotation_names'] == "NoClass" or float(row['annotations']) == -1.0:
            continue
            
        # 2. Extract relevant data
        ann_name = row['annotation_names']
        phtau_val = row['PhTau_cluster']
        treatment_val = row['treatment']
        
        # 3. Create descriptive label (e.g., "normal, PhTau+")
        phtau_status = "PhTau+" if phtau_val == '1' else "PhTau-"
        full_label = f"{ann_name}, {phtau_status}, {treatment_val}"
        
        # Store label and keep track of the original index to filter images later
        labels.append([row['annotations'], full_label])
        valid_indices.append(i)

# Update images and wavelet data to remove the "NoClass" entries
images = images[valid_indices]
wavelet_images = [wavelet_images[i] for i in valid_indices]
n_images = len(images)

# --- 5. Extract Unique Categories ---
# Create a sorted list of human-readable category names based on cluster ID
unique_pairs = sorted(set(tuple(x) for x in labels), key=lambda x: int(float((x[0]))))
categories = [pair[1] for pair in unique_pairs]

print(f"Loaded {n_images} images across {len(categories)} categories.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Create a DataFrame from the processed labels
df_plot = pd.DataFrame([label[1].split(", ") for label in labels], 
                       columns=["Annotation", "PhTau_Status", "Treatment"])

# 2. Combine Treatment and PhTau_Status to match your three specific conditions
# (dox-, dox+/PhTau-, dox+/PhTau+)
def map_condition(row):
    if row['Treatment'] == 'dox-':
        return 'dox-'
    return f"dox+/{row['PhTau_Status']}"

df_plot['Condition'] = df_plot.apply(map_condition, axis=1)

# 3. Create the plot
plt.figure(figsize=(12, 6))
sns.countplot(data=df_plot, x='Annotation', hue='Condition', 
              hue_order=['dox-', 'dox+/PhTau-', 'dox+/PhTau+'])

plt.title("Distribution of Annotations across Cell Conditions")
plt.xlabel("Annotation Category")
plt.ylabel("Count")
plt.legend(title="Condition")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a consistent colour dictionary for the six specific annotations
annotation_colours = {
    'big clusters': '#F1C40F',
    'dispersed': '#3498DB',
    'enlarged': '#E74C3C',
    'enlarged&dispersed': '#9B59B6',
    'normal': '#2ECC71',
    'other/atypical': '#95A5A6'
}

annotation_order = [
    'normal',
    'enlarged',
    'dispersed',
    'enlarged&dispersed',
    'big clusters',
    'other/atypical'
]

condition_order = ['dox-', 'dox+/PhTau-', 'dox+/PhTau+']

# Calculate annotation percentages within each condition
df_pct = (
    df_plot.groupby(['Condition', 'Annotation'])
    .size()
    .rename('count')
    .reset_index()
)
df_pct['percentage'] = (
    df_pct.groupby('Condition')['count']
    .transform(lambda x: 100 * x / x.sum())
)

# Create a 2x2 figure layout
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

# Map conditions to specific grid positions
grid_mapping = {
    'dox-': axes[0, 0],
    'dox+/PhTau-': axes[1, 0],
    'dox+/PhTau+': axes[1, 1]
}

for condition, ax in grid_mapping.items():
    # Select percentages for the current condition
    condition_data = (
        df_pct[df_pct['Condition'] == condition]
        .set_index('Annotation')['percentage']
        .reindex(annotation_order)
        .fillna(0)
    )
    values = condition_data.to_numpy()
    labels = condition_data.index
    colours = [annotation_colours[label] for label in labels]
    
    # Plot pie chart with a larger, readable font size for percentages
    ax.pie(
        values,
        colors=colours,
        autopct=lambda pct: f'{pct:.1f}%' if pct > 5.0 else '',
        startangle=90,
        counterclock=False,
        pctdistance=0.6,
        textprops={'fontsize': 11, 'color': 'black', 'weight': 'bold'},
        wedgeprops={'edgecolor': 'white', 'linewidth': 1}
    )
    ax.set_title(condition)
    ax.axis('equal')

# Use the top-right slot (axes[0, 1]) to host the shared legend
axes[0, 1].axis('off')
legend_handles = [
    plt.Line2D(
        [0], [0], 
        marker='o', 
        color='w', 
        label=annotation, 
        markerfacecolor=annotation_colours[annotation], 
        markersize=10
    ) 
    for annotation in annotation_order
]

axes[0, 1].legend(
    handles=legend_handles,
    title='Annotation',
    loc='center',
    frameon=False
)

fig.suptitle('Distribution of Annotations within each Condition', fontsize=16, y=0.98)
plt.savefig('Figures/distribution_pie.svg', format='svg', bbox_inches='tight')
plt.show()